# Prediksi Kelulusan Ujian Siswa (`pass_status`) dengan Gaussian Naïve Bayes
## Bagian A — Pra-pemrosesan Data

**Alur notebook**
1. Link data
2. Cerita tentang data
3. Skala pengukuran per kolom (nominal / ordinal / interval / rasio)
4. Gambaran nilai tiap kolom
5. Pemrosesan data yang dibutuhkan dan alasannya (5.1 – 5.9)
6. Hasil akhir dan penyimpanan untuk Bagian B (pelatihan model)

> **Prinsip anti-kebocoran data (*leakage*):** data dibagi lebih dulu, lalu semua statistik (batas IQR, dsb.) dihitung dari **data latih saja** dan diterapkan ke data uji. Oversampling hanya pada data latih.

## 0. Pengaturan
Semua path, konstanta, dan daftar kolom dikumpulkan di sini agar mudah diubah.

In [24]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# ---------- Pengaturan (ubah di sini bila perlu) ----------
PATH_DATA = '../data/student_exam_performance.csv'
   # data mentah (44 kolom)
PATH_LATIH   = "data/data_latih_proses.csv"          # keluaran: data latih (sudah oversampling)
PATH_UJI     = "data/data_uji_proses.csv"            # keluaran: data uji
TARGET       = "pass_status"
RANDOM_STATE = 42
UKURAN_UJI   = 0.20                                  # 80:20

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", None)

# 13 fitur terpilih, dikelompokkan menurut domain
FITUR_PER_KELOMPOK = {
    "Riwayat akademik":    ["previous_exam_score", "previous_gpa", "attendance_percentage", "assignment_completion_rate"],
    "Kebiasaan belajar":   ["study_hours_per_day", "self_study_hours", "practice_tests_completed", "exam_preparation_days"],
    "Gaya hidup / mental": ["sleep_hours", "stress_level", "exam_anxiety_level"],
    "Kondisi ujian":       ["exam_difficulty"],
    "Latar belakang":      ["family_income"],
}
FITUR         = [f for kolom in FITUR_PER_KELOMPOK.values() for f in kolom]
FITUR_ORDINAL = ["exam_difficulty", "family_income"]
FITUR_NUMERIK = [f for f in FITUR if f not in FITUR_ORDINAL]          # 11 kolom interval + rasio

# Kolom yang di-winsorizing (miring ke kanan, nilai ekstrem masih logis)
KOLOM_WINSOR = ["study_hours_per_day", "self_study_hours", "practice_tests_completed"]

# Pemetaan ordinal (tetap, tidak dipelajari dari data)
MAP_EXAM_DIFFICULTY = {"Easy": 1, "Medium": 2, "Hard": 3}
MAP_FAMILY_INCOME   = {"Low": 1, "Lower-Middle": 2, "Middle": 3, "Upper-Middle": 4, "High": 5}


def distribusi(seri):
    """Tabel jumlah dan persentase tiap kelas."""
    jumlah = seri.value_counts()
    return pd.DataFrame({"Jumlah": jumlah, "Persen (%)": (jumlah / jumlah.sum() * 100).round(2)})

## 1. Link Data

**Sumber dataset:** [ISI NAMA DATASET DI SINI](https://ISI-LINK-KAGGLE-ATAU-SUMBER-LAIN)

> ✏️ Ganti teks dan link di atas dengan sumber asli dataset (Kaggle atau lainnya).
> File yang dipakai notebook ini: `data/student_exam_performance.csv`.

## 2. Cerita tentang Data

**Data apa ini?** Data performa ujian siswa. Setiap baris adalah satu siswa dengan informasi latar belakang, riwayat akademik, kebiasaan belajar, pemanfaatan teknologi, kondisi gaya hidup dan mental, serta hasil ujian.

**Ukuran:** 100.000 baris dan 44 kolom; usia siswa 14–20 tahun (dicek pada sel di bawah).

**Tujuan klasifikasi:** memprediksi `pass_status` (**Pass** / **Fail**) dari kebiasaan belajar, riwayat akademik, kondisi keluarga (`family_income`), dan faktor lain, menggunakan **Gaussian Naïve Bayes**.

**Kondisi awal data**
- **Missing value:** 52.161 sel pada 7 kolom (`parent_education` 6,53%, `previous_gpa` 7,79%, `attendance_percentage` 9,86%, `notes_quality` 8,41%, `sleep_quality` 6,99%, `device_availability` 2,96%, `time_management_score` 9,61%). Hanya `previous_gpa` dan `attendance_percentage` yang termasuk fitur terpilih.
- **Duplikat:** tidak ada baris duplikat.
- **Distribusi target:** Pass 77,39% (77.389) dan Fail 22,61% (22.611), jadi **tidak seimbang**.

Angka-angka itu dikonfirmasi oleh sel berikut.

In [25]:
df = pd.read_csv(PATH_DATA)
df.index.name = "index_asli"

print(f"Ukuran data    : {df.shape[0]:,} baris x {df.shape[1]} kolom")
print(f"Rentang usia   : {df['age'].min()} - {df['age'].max()} tahun")
print(f"Baris duplikat : {df.duplicated().sum()}")

missing = df.isna().sum()
missing = missing[missing > 0]
print(f"\nNilai kosong: {missing.sum():,} sel pada {len(missing)} kolom")
print(pd.DataFrame({"Jumlah": missing, "Persen (%)": (missing / len(df) * 100).round(2)})
      .sort_values("Jumlah", ascending=False))

print(f"\nDistribusi target '{TARGET}':")
print(distribusi(df[TARGET]))

Ukuran data    : 100,000 baris x 44 kolom
Rentang usia   : 14 - 20 tahun
Baris duplikat : 0

Nilai kosong: 52,161 sel pada 7 kolom
                       Jumlah  Persen (%)
attendance_percentage    9863        9.86
time_management_score    9613        9.61
notes_quality            8407        8.41
previous_gpa             7793        7.79
sleep_quality            6994        6.99
parent_education         6526        6.53
device_availability      2965        2.96

Distribusi target 'pass_status':
             Jumlah  Persen (%)
pass_status                    
Pass          77389       77.39
Fail          22611       22.61


## 3. Skala Pengukuran per Kolom

Hanya 13 fitur dan 1 target yang dipakai (alasan pemilihan ada di bagian 5.7), jadi tabel ini mencakup 14 kolom.

- **Nominal:** kategori tanpa urutan.
- **Ordinal:** kategori berurutan, jarak antar tingkat tidak tentu sama.
- **Interval:** numerik dengan jarak bermakna, tanpa nol mutlak.
- **Rasio:** numerik dengan nol mutlak.


In [26]:
data = df[FITUR + [TARGET]].copy()          # 13 fitur + 1 target, data mentah
assert data.shape[1] == 14

SKALA = [
    # (kolom, skala, alasan)
    (TARGET, "Nominal", "Dua kategori (Pass/Fail) yang dipakai sebagai label kelas; tidak dipakai sebagai nilai berurutan."),
    ("exam_difficulty", "Ordinal", "Easy < Medium < Hard berurutan, tetapi jarak antar tingkat tidak tentu sama."),
    ("family_income", "Ordinal", "Low < Lower-Middle < Middle < Upper-Middle < High; kategori berurutan, jarak tidak sama."),
    ("previous_exam_score", "Interval", "Skor 0-100; nol bukan 'tanpa kemampuan', jadi tanpa nol mutlak (bisa diperdebatkan sebagai rasio)."),
    ("previous_gpa", "Interval", "Skala IPK 0-4; nilai 0 bukan ketiadaan prestasi."),
    ("stress_level", "Interval", "Rating skala 5-10 pada data; sebenarnya rating (bisa dianggap ordinal), diperlakukan interval karena dipakai sebagai angka berjarak sama."),
    ("exam_anxiety_level", "Interval", "Rating skala 1-10 tanpa nol mutlak; sama seperti stress_level, diperlakukan interval."),
    ("attendance_percentage", "Rasio", "Persentase kehadiran; 0% berarti tidak pernah hadir (nol mutlak)."),
    ("assignment_completion_rate", "Rasio", "Persentase tugas selesai; 0% berarti tidak ada tugas selesai."),
    ("study_hours_per_day", "Rasio", "Jam belajar per hari; 0 berarti tidak belajar."),
    ("self_study_hours", "Rasio", "Jam belajar mandiri; 0 berarti tidak ada."),
    ("practice_tests_completed", "Rasio", "Cacah (bilangan bulat) tes latihan; 0 berarti tidak ada."),
    ("exam_preparation_days", "Rasio", "Cacah hari persiapan ujian; 0 berarti tidak ada."),
    ("sleep_hours", "Rasio", "Jam tidur; 0 berarti tidak tidur."),
]
tabel_skala = pd.DataFrame(SKALA, columns=["Kolom", "Skala", "Alasan / catatan"])
assert set(tabel_skala["Kolom"]) == set(data.columns), "Kolom tabel skala tidak sesuai data!"

print("Jumlah kolom per skala:", tabel_skala["Skala"].value_counts().to_dict())
tabel_skala

Jumlah kolom per skala: {'Rasio': 7, 'Interval': 4, 'Ordinal': 2, 'Nominal': 1}


,Kolom,Skala,Alasan / catatan
0,pass_status,Nominal,Dua kategori (Pass/Fail) yang dipakai sebagai label kelas; tidak dipakai sebagai nilai berurutan.
1,exam_difficulty,Ordinal,"Easy < Medium < Hard berurutan, tetapi jarak antar tingkat tidak tentu sama."
2,family_income,Ordinal,"Low < Lower-Middle < Middle < Upper-Middle < High; kategori berurutan, jarak tidak sama."
3,previous_exam_score,Interval,"Skor 0-100; nol bukan 'tanpa kemampuan', jadi tanpa nol mutlak (bisa diperdebatkan sebagai rasio)."
4,previous_gpa,Interval,Skala IPK 0-4; nilai 0 bukan ketiadaan prestasi.
5,stress_level,Interval,"Rating skala 5-10 pada data; sebenarnya rating (bisa dianggap ordinal), diperlakukan interval karena dipakai sebagai angka berjarak sama."
6,exam_anxiety_level,Interval,"Rating skala 1-10 tanpa nol mutlak; sama seperti stress_level, diperlakukan interval."
7,attendance_percentage,Rasio,Persentase kehadiran; 0% berarti tidak pernah hadir (nol mutlak).
8,assignment_completion_rate,Rasio,Persentase tugas selesai; 0% berarti tidak ada tugas selesai.
9,study_hours_per_day,Rasio,Jam belajar per hari; 0 berarti tidak belajar.


## 4. Gambaran Nilai Tiap Kolom
Nilai minimum, rata-rata, simpangan baku, dan maksimum untuk 11 kolom interval/rasio (data mentah, sebelum pembersihan). Rentang yang jauh berbeda antar kolom menjadi dasar keputusan di bagian 5.6.
Statistik deskriptif ini digunakan untuk mendeteksi masalah pada data sebelum diproses. Nilai maksimum yang jauh melampaui rata-rata pada kolom study_hours_per_day, self_study_hours, dan practice_tests_completed (4–5 kali lipat rata-ratanya) menandakan adanya nilai ekstrem sehingga perlu winsorizing. Perbedaan rentang nilai yang besar antar kolom — seperti previous_gpa (1–4) dan previous_exam_score (30–100) — menjadi alasan dilakukannya normalisasi. Selain itu, dua kolom (previous_gpa dan attendance_percentage) memiliki nilai kosong yang perlu ditangani dengan imputasi. Terakhir, distribusi target yang tidak seimbang (Pass 77% vs Fail 23%) mengharuskan dilakukannya oversampling pada data latih agar model tidak bias ke kelas mayoritas.

In [27]:
print(data[FITUR_NUMERIK].describe().T[["min", "mean", "std", "max"]].round(2))
print("\nKategori pada kolom ordinal:")
for kol in FITUR_ORDINAL:
    print(f"  {kol}: {data[kol].unique().tolist()}")

                              min   mean    std    max
previous_exam_score         30.00  69.53  11.38  100.0
previous_gpa                 1.00   2.78   0.48    4.0
attendance_percentage       46.34  84.62   7.92  100.0
assignment_completion_rate  20.00  69.72  12.13  100.0
study_hours_per_day          0.50   3.00   1.72   12.0
self_study_hours             0.25   2.09   1.27   10.0
practice_tests_completed     0.00   5.51   2.50   20.0
exam_preparation_days        1.00  15.50   8.64   30.0
sleep_hours                  3.00   7.00   1.20   11.0
stress_level                 5.00   9.38   1.08   10.0
exam_anxiety_level           1.00   8.93   1.35   10.0

Kategori pada kolom ordinal:
  exam_difficulty: ['Medium', 'Easy', 'Hard']
  family_income: ['Middle', 'High', 'Lower-Middle', 'Upper-Middle', 'Low']


## 5. Pemrosesan Data yang Dibutuhkan dan Alasannya

Urutan: **duplikat & inkonsistensi → split → missing value → outlier → kategorikal ke numerik → normalisasi → seleksi fitur → data tidak seimbang → alasan khusus Gaussian Naïve Bayes.** Setiap sub-bagian ditulis dengan pola *masalah → tindakan → alasan*.

### 5.1 Pengecekan duplikat dan inkonsistensi

**Masalah:** Baris ganda membuat satu siswa terhitung dua kali dan bisa tersebar ke data latih *dan* data uji (bocor). Kategori yang tidak dikenal atau nilai di luar rentang logis (misalnya kehadiran > 100%) merusak statistik. Pada eksplorasi awal tidak ada duplikat.

**Tindakan:** Cek duplikat, kategori valid, rentang logis, dan tipe data. Dua cek logika tambahan (belajar mandiri > jam belajar, belajar + tidur > 24 jam) bersifat informatif dan tidak menghentikan notebook.

**Alasan:** Pengecekan dilakukan **sebelum** split agar baris identik tidak terpisah ke dua sisi. Kategori dicek **sebelum** encoding supaya nilai tak dikenal tidak diam-diam menjadi NaN.

In [28]:
RENTANG_VALID = {
    "previous_exam_score": (0, 100), "previous_gpa": (0, 4),
    "attendance_percentage": (0, 100), "assignment_completion_rate": (0, 100),
    "study_hours_per_day": (0, 24), "self_study_hours": (0, 24),
    "practice_tests_completed": (0, np.inf), "exam_preparation_days": (0, np.inf),
    "sleep_hours": (0, 24), "stress_level": (1, 10), "exam_anxiety_level": (1, 10),
}
KATEGORI_VALID = {
    "exam_difficulty": set(MAP_EXAM_DIFFICULTY),
    "family_income": set(MAP_FAMILY_INCOME),
    TARGET: {"Pass", "Fail"},
}

print("Baris duplikat:", data.duplicated().sum())

for kol, valid in KATEGORI_VALID.items():
    tak_dikenal = set(data[kol].dropna().unique()) - valid
    assert not tak_dikenal, f"{kol}: kategori tak dikenal {tak_dikenal}"
print("Kategori tak dikenal: tidak ada")


print("Tipe data:", data.dtypes.astype(str).value_counts().to_dict())

Baris duplikat: 0
Kategori tak dikenal: tidak ada
Tipe data: {'float64': 8, 'int64': 3, 'object': 3}


### 5.2 Pembagian data (80:20, stratified)

**Masalah:** Langkah berikutnya (batas IQR, oversampling) mempelajari sesuatu dari data. Jika data uji ikut terlihat, skor uji menjadi terlalu optimis. Kelas target juga tidak seimbang (77:23), sehingga pembagian acak biasa bisa menghasilkan proporsi kelas yang berbeda antara latih dan uji.

**Tindakan:** Bagi data 80% latih dan 20% uji dengan `stratify=pass_status` dan `random_state=42`. Pembagian dilakukan **sebelum** penanganan missing value, outlier, dan oversampling.

**Alasan:** Dengan split lebih dulu, semua statistik yang dipelajari hanya berasal dari data latih, sehingga data uji benar-benar "belum pernah dilihat". *Stratified* menjaga proporsi Pass/Fail yang sama di kedua bagian. Rasio 80:20 cukup: dengan ±83.000 baris bersih, data uji tetap belasan ribu baris. `random_state` membuat hasil bisa diulang.

In [29]:
data_latih, data_uji = train_test_split(
    data, test_size=UKURAN_UJI, stratify=data[TARGET], random_state=RANDOM_STATE
)
data_latih, data_uji = data_latih.copy(), data_uji.copy()

print(f"Data latih: {data_latih.shape} | Data uji: {data_uji.shape}")
print(pd.concat({"Latih": distribusi(data_latih[TARGET]), "Uji": distribusi(data_uji[TARGET])}, axis=1))

Data latih: (80000, 14) | Data uji: (20000, 14)
             Latih               Uji           
            Jumlah Persen (%) Jumlah Persen (%)
pass_status                                    
Pass         61911      77.39  15478      77.39
Fail         18089      22.61   4522      22.61


### 5.3 Penanganan missing value

**Masalah:** Dua dari 13 fitur memiliki nilai kosong: `previous_gpa` (±7,8%) dan `attendance_percentage` (±9,9%). `GaussianNB` di scikit-learn tidak menerima NaN.

**Tindakan:** Buang baris yang memiliki nilai kosong, pada data latih dan data uji secara terpisah (indeks asli tidak di-reset). Proporsi kelas sesudah pembuangan dibandingkan.

**Alasan:** Data yang tersisa masih sangat besar (puluhan ribu baris). Imputasi satu nilai (rata-rata/median) akan mengecilkan varians dan menggeser rata-rata, padahal Gaussian Naïve Bayes hanya memakai rata-rata dan varians tiap fitur per kelas, jadi estimasinya ikut terdistorsi. Pembuangan baris tidak mempelajari statistik apa pun dari data, sehingga aman dilakukan pada data uji juga. Proporsi Pass/Fail yang nyaris tidak berubah menunjukkan pembuangan tidak menghilangkan satu kelas lebih banyak daripada yang lain.

In [30]:
kosong = pd.DataFrame({"Latih": data_latih.isna().sum(), "Uji": data_uji.isna().sum()})
print("Nilai kosong per kolom (hanya yang memiliki NaN):")
print(kosong[kosong.sum(axis=1) > 0])

n_awal = {"latih": len(data_latih), "uji": len(data_uji)}
data_latih = data_latih.dropna().copy()     # indeks asli TIDAK di-reset
data_uji = data_uji.dropna().copy()

print()
for nama, bagian in [("latih", data_latih), ("uji", data_uji)]:
    n0 = n_awal[nama]
    print(f"Data {nama:5s}: {n0:,} -> {len(bagian):,} baris (dibuang {n0 - len(bagian):,} = {(n0 - len(bagian)) / n0 * 100:.2f}%)")

print("\nDistribusi pass_status sesudah dropna:")
print(pd.concat({"Latih": distribusi(data_latih[TARGET]), "Uji": distribusi(data_uji[TARGET])}, axis=1))

Nilai kosong per kolom (hanya yang memiliki NaN):
                       Latih   Uji
previous_gpa            6197  1596
attendance_percentage   7896  1967

Data latih: 80,000 -> 66,520 baris (dibuang 13,480 = 16.85%)
Data uji  : 20,000 -> 16,605 baris (dibuang 3,395 = 16.98%)

Distribusi pass_status sesudah dropna:
             Latih               Uji           
            Jumlah Persen (%) Jumlah Persen (%)
pass_status                                    
Pass         51456      77.35  12816      77.18
Fail         15064      22.65   3789      22.82


### 5.4 Penanganan outlier

**Masalah:** Batas IQR (1,5×) dihitung dari data latih dan dibandingkan dengan skewness. Ada tiga pola:
- **Ekor kanan nyata** pada `study_hours_per_day`, `self_study_hours`, `practice_tests_completed` (skewness positif, ±1,8–3% nilai di atas batas atas).
- **Penumpukan di nilai maksimum skala** pada `stress_level` dan `exam_anxiety_level` (skewness negatif sekitar −1,4 s.d. −1,6). Nilai 10 adalah batas atas skala, bukan nilai ekstrem.
- **Kolom lain** hanya memiliki sedikit nilai di luar batas (< ±0,7%) dan masih berada pada rentang yang masuk akal.

**Tindakan:** *Winsorizing* (clipping ke batas IQR) hanya pada 3 kolom ekor kanan. Batas dihitung dari data latih lalu diterapkan ke data latih dan uji. Untuk kolom cacah, batas dibulatkan ke bilangan bulat agar tipe data tetap integer. Kolom lain dibiarkan.

**Alasan:** Gaussian Naïve Bayes hanya mengestimasi rata-rata dan varians, dan keduanya sangat peka terhadap nilai ekstrem. Nilai ekstrem di sini masih mungkin terjadi (misalnya belajar 12 jam sehari), jadi baris tidak dibuang; nilainya cukup dibatasi. Batas dari data latih saja mencegah kebocoran dari data uji. Pada `stress_level` dan `exam_anxiety_level` tidak ada yang perlu dipotong karena nilai maksimum skala itu valid.

In [31]:
def batas_iqr(seri):
    q1, q3 = seri.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


baris = []
for kol in FITUR_NUMERIK:
    s = data_latih[kol]
    bawah, atas = batas_iqr(s)
    n_bawah, n_atas = int((s < bawah).sum()), int((s > atas).sum())
    baris.append({
        "kolom": kol, "min": s.min(), "max": s.max(),
        "batas_bawah": round(bawah, 2), "batas_atas": round(atas, 2),
        "n_bawah": n_bawah, "n_atas": n_atas,
        "persen_outlier": round((n_bawah + n_atas) / len(s) * 100, 2),
        "skewness": round(s.skew(), 2),
    })
tabel_outlier = pd.DataFrame(baris).set_index("kolom")
tabel_outlier

,min,max,batas_bawah,batas_atas,n_bawah,n_atas,persen_outlier,skewness
kolom,,,,,,,,
previous_exam_score,30.00,100.0,38.67,100.39,214,0,0.32,-0.01
previous_gpa,1.00,4.0,1.49,4.08,214,0,0.32,-0.02
attendance_percentage,46.34,100.0,62.80,106.64,247,0,0.37,-0.17
assignment_completion_rate,20.00,100.0,37.02,102.43,219,0,0.33,0.01
study_hours_per_day,0.50,12.0,-1.57,7.20,0,1719,2.58,1.13
self_study_hours,0.25,10.0,-1.19,5.10,0,1936,2.91,1.29
practice_tests_completed,0.00,20.0,-0.50,11.50,0,1142,1.72,0.54
exam_preparation_days,1.00,30.0,-14.50,45.50,0,0,0.00,-0.00
sleep_hours,3.00,11.0,3.73,10.25,214,211,0.64,0.00


**Winsorizing** pada 3 kolom. Tabel kedua membandingkan skewness data latih sebelum dan sesudah clipping.

batas bawah itu digunakan kaya kalo ada satu gars terus di kaya battas bawah nya nya kaya sekian kalo atas nya sekian jadi kalop lewat harus di turunin sampe ke batas nya, contoh org belajar kan gamungkin 12 jam jadi nanti di potong ke batas atas nya bair nilai nya wajar


In [32]:
skew_sebelum = data_latih[KOLOM_WINSOR].skew()

batas_winsor = {}
for kol in KOLOM_WINSOR:
    bawah, atas = batas_iqr(data_latih[kol])
    if pd.api.types.is_integer_dtype(data_latih[kol]):     # kolom cacah: batas dibuat bilangan bulat
        bawah, atas = np.ceil(bawah), np.floor(atas)
    batas_winsor[kol] = (bawah, atas)

ringkasan = []
for nama, bagian in [("latih", data_latih), ("uji", data_uji)]:
    for kol, (b, a) in batas_winsor.items():
        asli = bagian[kol].copy()
        bagian[kol] = bagian[kol].clip(lower=b, upper=a).astype(asli.dtype)
        ringkasan.append({
            "data": nama, "kolom": kol, "batas_bawah": round(b, 2), "batas_atas": round(a, 2),
            "jumlah_nilai_berubah": int((asli != bagian[kol]).sum()),
            "max_sebelum": asli.max(), "max_sesudah": bagian[kol].max(),
        })
print(pd.DataFrame(ringkasan).to_string(index=False))

print("\nSkewness data latih:")
print(pd.DataFrame({"sebelum": skew_sebelum, "sesudah": data_latih[KOLOM_WINSOR].skew()}).round(2))

 data                    kolom  batas_bawah  batas_atas  jumlah_nilai_berubah  max_sebelum  max_sesudah
latih      study_hours_per_day        -1.57         7.2                  1719         12.0        7.195
latih         self_study_hours        -1.19         5.1                  1936         10.0        5.095
latih practice_tests_completed        -0.00        11.0                  1142         20.0       11.000
  uji      study_hours_per_day        -1.57         7.2                   429         12.0        7.195
  uji         self_study_hours        -1.19         5.1                   486         10.0        5.095
  uji practice_tests_completed        -0.00        11.0                   309         18.0       11.000

Skewness data latih:
                          sebelum  sesudah
study_hours_per_day          1.13     0.76
self_study_hours             1.29     0.81
practice_tests_completed     0.54     0.31


### 5.5 Transformasi kategorikal ke numerik (ordinal dan nominal)

**Masalah:** Dua fitur ordinal bertipe teks (`exam_difficulty`, `family_income`), padahal `GaussianNB` hanya menerima angka. Tidak ada fitur nominal; satu-satunya kolom nominal adalah target `pass_status` (Pass/Fail).

**Tindakan:**

| Kolom | Pemetaan |
|---|---|
| `exam_difficulty` | Easy = 1, Medium = 2, Hard = 3 |
| `family_income` | Low = 1, Lower-Middle = 2, Middle = 3, Upper-Middle = 4, High = 5 |

Target dibiarkan sebagai label teks `Pass` / `Fail`.

**Alasan:** *Ordinal encoding* mempertahankan urutan tingkat (1 < 2 < 3). *One-hot encoding* sengaja tidak dipakai: kolom 0/1 jelas tidak berdistribusi normal, sehingga tidak cocok dengan asumsi Gaussian, dan jumlah fitur menjadi bertambah. Pemetaan ini tetap (tidak dipelajari dari data), jadi tidak ada risiko kebocoran dan aman diterapkan ke data latih maupun uji. scikit-learn menerima label kelas berupa teks, sehingga target tidak perlu diubah.

In [33]:
for bagian in (data_latih, data_uji):
    bagian["exam_difficulty"] = bagian["exam_difficulty"].map(MAP_EXAM_DIFFICULTY)
    bagian["family_income"] = bagian["family_income"].map(MAP_FAMILY_INCOME)
    assert bagian[FITUR_ORDINAL].isna().sum().sum() == 0, "NaN muncul setelah encoding!"
    assert bagian["exam_difficulty"].between(1, 3).all() and bagian["family_income"].between(1, 5).all()

for kol in FITUR_ORDINAL:
    print(f"{kol}: nilai pada data latih = {sorted(data_latih[kol].unique().tolist())}")
print("Verifikasi lolos: tidak ada NaN, semua nilai dalam rentang pemetaan.")

exam_difficulty: nilai pada data latih = [1, 2, 3]
family_income: nilai pada data latih = [1, 2, 3, 4, 5]
Verifikasi lolos: tidak ada NaN, semua nilai dalam rentang pemetaan.


### 5.6 Normalisasi / standarisasi

**Masalah:** Skala antar fitur sangat berbeda (tabel bagian 4): skor 0–100, IPK 1–4, jam tidur 3–11, hari persiapan 1–30, dan seterusnya.

**Tindakan:** **Tidak dilakukan** (tidak ada z-score maupun min-max).

**Alasan:** Gaussian Naïve Bayes menghitung rata-rata dan varians **tiap fitur secara terpisah untuk tiap kelas**, lalu menggabungkan peluangnya. Mengubah skala sebuah fitur secara linear (z-score, min-max) mengubah rata-rata dan simpangan baku kedua kelas dengan cara yang sama, sehingga perbandingan peluang antar kelas, dan keputusan akhir, praktis tidak berubah. Satu-satunya pengecualian kecil adalah `var_smoothing`, yang bergantung pada varians terbesar. Dengan tidak menstandarkan, nilai asli tetap mudah ditafsirkan. Kalau Bagian B nanti membandingkan model yang peka skala (KNN, SVM, regresi logistik), standarisasi perlu ditambahkan dengan parameter dari data latih saja.

### 5.7 Seleksi fitur

**Masalah:** Dataset memiliki 44 kolom, dan tidak semuanya layak dipakai sebagai fitur:
- `exam_score`, `performance_grade`, `performance_level`, `questions_correct` dihasilkan dari **hasil ujian itu sendiri**. Contohnya, `Fail` identik dengan grade `F` (dicek di sel berikut). Memakainya sebagai fitur membuat model "mencontek" jawaban (*data leakage*).
- `student_id` hanyalah pengenal tanpa makna prediktif.
- 25 kolom lain tidak dipilih untuk menjaga model tetap ringkas dan berfokus pada lima kelompok faktor di bawah. Lima di antaranya (`parent_education`, `notes_quality`, `sleep_quality`, `device_availability`, `time_management_score`) juga memiliki missing value; membuangnya menghindari kehilangan baris tambahan. Atribut demografis seperti `gender` dan `urban_rural` otomatis tidak dipakai model.

**Tindakan:** Pertahankan 13 fitur:

| Kelompok | Fitur |
|---|---|
| Riwayat akademik | `previous_exam_score`, `previous_gpa`, `attendance_percentage`, `assignment_completion_rate` |
| Kebiasaan belajar | `study_hours_per_day`, `self_study_hours`, `practice_tests_completed`, `exam_preparation_days` |
| Gaya hidup / mental | `sleep_hours`, `stress_level`, `exam_anxiety_level` |
| Kondisi ujian | `exam_difficulty` |
| Latar belakang | `family_income` |

**Alasan:** Seleksi didasarkan pada pengetahuan domain dan pencegahan *leakage*, **bukan** pada statistik data, sehingga aman dikerjakan di awal (sebelum split) tanpa membocorkan data uji. Fitur yang lebih sedikit dan beragam juga membantu asumsi independensi antar fitur pada Naïve Bayes (dicek di 5.9).

In [34]:
KOLOM_BOCOR_ID = ["exam_score", "performance_grade", "performance_level", "questions_correct", "student_id"]
KOLOM_TIDAK_DIPAKAI = [
    "age", "gender", "education_level", "school_type", "parent_education", "urban_rural",
    "class_participation", "study_consistency", "study_environment", "study_method",
    "revision_frequency", "notes_quality", "private_tuition", "time_management_score",
    "sleep_quality", "daily_screen_time", "physical_activity_hours", "break_frequency",
    "motivation_level", "internet_access", "device_availability",
    "educational_app_usage", "online_learning_hours", "online_course_hours",
    "questions_attempted",
]

# Validasi: setiap kolom terdaftar tepat satu kali dan cocok dengan CSV
semua = FITUR + [TARGET] + KOLOM_BOCOR_ID + KOLOM_TIDAK_DIPAKAI
assert len(semua) == len(set(semua)), "Ada kolom terdaftar ganda!"
assert set(semua) == set(df.columns), "Kolom tidak sesuai dengan dataset!"

print(pd.DataFrame({
    "Peran": ["Fitur (X)", "Target (y)", "Dibuang: bocor / ID", "Dibuang: tidak dipilih", "Total"],
    "Jumlah kolom": [len(FITUR), 1, len(KOLOM_BOCOR_ID), len(KOLOM_TIDAK_DIPAKAI), len(semua)],
}).to_string(index=False))

print("\nBukti leakage: 'Fail' identik dengan grade 'F':",
      bool((df["performance_grade"].eq("F") == df[TARGET].eq("Fail")).all()))
print("Kolom tidak dipilih yang memiliki NaN:", [k for k in KOLOM_TIDAK_DIPAKAI if df[k].isna().any()])

                 Peran  Jumlah kolom
             Fitur (X)            13
            Target (y)             1
   Dibuang: bocor / ID             5
Dibuang: tidak dipilih            25
                 Total            44

Bukti leakage: 'Fail' identik dengan grade 'F': True
Kolom tidak dipilih yang memiliki NaN: ['parent_education', 'notes_quality', 'time_management_score', 'sleep_quality', 'device_availability']


### 5.8 Penanganan data tidak seimbang (oversampling)

**Masalah:** Pada data latih, Pass ±77% dan Fail ±23%. Model cenderung condong ke kelas mayoritas, sedangkan Fail justru kelas yang penting dikenali.

**Tindakan:** *Random oversampling*: baris kelas minoritas (Fail) diambil ulang secara acak **dengan pengembalian** sampai jumlahnya sama dengan Pass (50:50). Dilakukan **setelah** split dan **hanya pada data latih**; data uji tidak diubah.

**Alasan:** Jika oversampling dilakukan sebelum split, salinan baris yang sama bisa masuk ke data uji sehingga skor menjadi palsu. Data uji dibiarkan asli (±77:23) supaya metrik evaluasi mencerminkan kondisi sebenarnya. Catatan khusus untuk Gaussian Naïve Bayes ada di 5.9.

In [35]:
dist_sebelum = distribusi(data_latih[TARGET])

jumlah = data_latih[TARGET].value_counts()
kelas_mayor, kelas_minor = jumlah.idxmax(), jumlah.idxmin()
mayor = data_latih[data_latih[TARGET] == kelas_mayor]
minor = data_latih[data_latih[TARGET] == kelas_minor]

minor_naik = resample(minor, replace=True, n_samples=len(mayor), random_state=RANDOM_STATE)
data_latih_os = pd.concat([mayor, minor_naik]).sample(frac=1, random_state=RANDOM_STATE)   # acak urutan

print(f"Kelas mayoritas: {kelas_mayor} | Kelas minoritas: {kelas_minor}")
print(pd.concat({"Sebelum": dist_sebelum, "Sesudah": distribusi(data_latih_os[TARGET])}, axis=1))
print(f"\nData uji tidak diubah: {len(data_uji):,} baris")

Kelas mayoritas: Pass | Kelas minoritas: Fail
            Sebelum            Sesudah           
             Jumlah Persen (%)  Jumlah Persen (%)
pass_status                                      
Pass          51456      77.35   51456       50.0
Fail          15064      22.65   51456       50.0

Data uji tidak diubah: 16,605 baris


## 6. Hasil Akhir dan Penyimpanan
Data dipisah menjadi fitur (X) dan target (y), disimpan ke CSV untuk Bagian B, lalu dibaca ulang untuk memastikan isinya sama persis dengan yang ada di memori.

In [37]:
X_train, y_train = data_latih_os.drop(columns=TARGET), data_latih_os[TARGET]
X_test, y_test = data_uji.drop(columns=TARGET), data_uji[TARGET]
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test : {X_test.shape} | y_test : {y_test.shape}")

os.makedirs(os.path.dirname(PATH_LATIH), exist_ok=True)
data_latih_os.to_csv(PATH_LATIH, index=True, index_label="index_asli")
data_uji.to_csv(PATH_UJI, index=True, index_label="index_asli")

# Verifikasi: baca ulang dan bandingkan
latih_b = pd.read_csv(PATH_LATIH, index_col="index_asli")
uji_b = pd.read_csv(PATH_UJI, index_col="index_asli")
assert latih_b.isna().sum().sum() == 0 and uji_b.isna().sum().sum() == 0
assert list(latih_b.columns) == list(uji_b.columns)
pd.testing.assert_frame_equal(latih_b, data_latih_os, check_dtype=False)
pd.testing.assert_frame_equal(uji_b, data_uji, check_dtype=False)
print(f"\nTersimpan dan terverifikasi: {PATH_LATIH} dan {PATH_UJI}")

print("\n5 baris teratas data latih akhir:")
data_latih_os.head(5)

X_train: (102912, 13) | y_train: (102912,)
X_test : (16605, 13) | y_test : (16605,)



Tersimpan dan terverifikasi: data/data_latih_proses.csv dan data/data_uji_proses.csv

5 baris teratas data latih akhir:


,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
index_asli,,,,,,,,,,,,,,
33919,59.68,2.34,85.61,76.83,3.76,2.41,6,26,6.54,7,7.17,1,4,Pass
44758,89.69,3.96,70.09,40.40,3.06,2.05,5,30,5.19,8,9.38,3,4,Pass
77911,55.98,2.16,75.46,76.48,1.86,1.56,5,11,8.93,10,8.82,3,2,Fail
78174,74.53,3.03,100.00,80.71,3.78,2.09,5,16,8.76,6,9.13,1,4,Pass
72558,56.88,2.29,69.01,76.03,4.36,3.55,5,5,5.39,10,9.46,3,4,Fail
